
# Notebook 03 · Multimodal en MongoDB Atlas con CLIP

Este notebook extiende la lógica del Notebook 02 a un escenario **multimodal** usando **MongoDB Atlas Vector Search**:

- **texto → texto** con **MiniLM**
- **texto ↔ imagen** con **CLIP**
- **imagen → imagen** con **CLIP**
- **imagen → texto** con **CLIP**

> Idea central: usamos **dos espacios vectoriales**.
>
> - **MiniLM** para recuperación semántica pura de texto.
> - **CLIP** para cruces entre texto e imagen, y para imagen→imagen.


## 1. Instalación de dependencias

In [1]:
!pip -q install pymongo[srv] sentence-transformers transformers pillow requests openai pandas numpy nltk

## 2. Importaciones y configuración inicial

In [2]:
import os
import io
import json
import math
import uuid
import time
import base64
import getpass
import requests
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime
from typing import List, Optional, Dict, Any

from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from openai import OpenAI

import torch

# Desactivamos el cálculo de gradientes para optimizar la RAM en Colab
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

## 3. Variables de conexión

In [3]:
from google.colab import userdata
MONGODB_URI = userdata.get('MONGO_URI')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

DB_NAME = "cookflow"

VECTOR_INDEX_NAME_TEXT   = "vector_index_texto"
VECTOR_INDEX_NAME_IMAGES = "vector_index_imagenes"

LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_MODEL = "llama-3.1-8b-instant"

print("[OK] Configuracion cargada desde secretos de Colab.")

[OK] Configuracion cargada desde secretos de Colab.


## 4. Conexión a MongoDB Atlas y carga de modelos

In [4]:
# Conexión al servidor Atlas
client = MongoClient(MONGODB_URI)
db = client[DB_NAME]

# MAPEO OFICIAL: Enlazamos las variables a tus colecciones de base de datos reales
chunks_col = db["chunks_embeddings"]        # Tu colección oficial de chunks de texto
images_col = db["imagenes_embeddings"]      # Tu colección oficial de imágenes del catálogo
history_collection = db["auditoria_rag"] # Colección única autorizada para logs de auditoría

# Carga de Modelos: MiniLM para texto puro y CLIP para la magia multimodal (Texto e Imagen)
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("[OK] Conexión establecida con 'cookflow' y modelos multimodales cargados con éxito.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[OK] Conexión establecida con 'cookflow' y modelos multimodales cargados con éxito.


## 5. Verificación rápida de colecciones

In [5]:
summary = []
# Evaluamos las colecciones reales asignadas al motor multimodal y auditoría
for name in ["chunks_embeddings", "imagenes_embeddings", "auditoria_rag"]:
    try:
        summary.append({
            "Colección Culinaria": name,
            "Documentos Registrados": db[name].count_documents({})
        })
    except Exception as e:
        summary.append({
            "Colección Culinaria": name,
            "Documentos Registrados": f"ERROR: {e}"
        })

# Desplegamos la tabla de control en Pandas
pd.DataFrame(summary)

,Colección Culinaria,Documentos Registrados
0,chunks_embeddings,242
1,imagenes_embeddings,50
2,auditoria_rag,27


## 6. Funciones base de embeddings y utilidades

In [6]:
def embed_text_minilm(text: str) -> List[float]:
    return embedding_model.encode(text, normalize_embeddings=True).tolist()


def embed_text_clip(text: str) -> List[float]:
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)
    text_features = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features[0].cpu().numpy().tolist()


def load_image_from_path_or_url(image_source: str) -> Image.Image:
    if image_source.startswith("http://") or image_source.startswith("https://"):
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(image_source, timeout=30, headers=headers)
        resp.raise_for_status()
        return Image.open(io.BytesIO(resp.content)).convert("RGB")
    return Image.open(image_source).convert("RGB")


def embed_image_clip(image_source: str) -> List[float]:
    image = load_image_from_path_or_url(image_source)
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
    image_features = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    return image_features[0].cpu().numpy().tolist()


def sentence_chunking(text: str, max_sentences: int = 3) -> List[str]:
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks

## 7. Datos multimodales

In [7]:
# Leemos todas las recetas que tienen imágenes con URL definida
recetas_con_imagenes = list(db.recetas.find(
    {"imagenes": {"$exists": True, "$ne": []}},
    {"_id": 1, "titulo": 1, "tags": 1, "imagenes": 1}
))

# Construimos la lista de imágenes a vectorizar
IMAGES_TO_PROCESS = []
for receta in recetas_con_imagenes:
    for img in receta.get("imagenes", []):
        if img.get("url", "").startswith("http"):
            IMAGES_TO_PROCESS.append({
                "receta_id": str(receta["_id"]),
                "titulo_receta": receta["titulo"],
                "url": img["url"],
                "descripcion": img.get("descripcion", ""),
                "etiquetas": receta.get("tags", [])
            })

print(f"Imágenes a vectorizar desde recetas reales: {len(IMAGES_TO_PROCESS)}")
for img in IMAGES_TO_PROCESS[:5]:
    print(f"  - {img['titulo_receta']}: {img['url']}")

Imágenes a vectorizar desde recetas reales: 50
  - Ajiaco Bogotano: https://i.blogs.es/8cb22c/ajiaco_colombiano_aguacate-min/1366_2000.jpeg
  - Bandeja Paisa Tradicional: https://i.blogs.es/bb0cca/bandeja_paisa/450_1000.jpg
  - Pollo al Curry Express: https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRJ4OcyG8uEH9CZGa5CPY9GrNXOJI69VyuSwA&s
  - Sancocho de Gallina Costeno: https://i.ytimg.com/vi/Y-g63sNgyiw/maxresdefault.jpg
  - Ceviche de Camaron: https://imag.bonviveur.com/ceviche-de-camaron.jpg


## 8. Función de ingesta e indexación multimodal

In [8]:
def ingest_images_from_recetas(images_list: List[Dict], batch_size: int = 10):
    total = len(images_list)
    exitosas = 0
    fallidas = 0

    for i, img_data in enumerate(images_list):
        try:
            # Generamos el embedding CLIP de la imagen real
            embedding = embed_image_clip(img_data["url"])

            # Upsert en imagenes_embeddings usando receta_id + url como clave única
            images_col.update_one(
                {
                    "receta_id": img_data["receta_id"],
                    "url": img_data["url"]
                },
                {
                    "$set": {
                        "receta_id": img_data["receta_id"],
                        "titulo_receta": img_data["titulo_receta"],
                        "url": img_data["url"],
                        "descripcion": img_data["descripcion"],
                        "etiquetas": img_data["etiquetas"],
                        "embedding_clip": embedding,
                        "modelo": "clip-vit-base-patch32",
                        "fecha_ingesta": datetime.utcnow()
                    }
                },
                upsert=True
            )
            exitosas += 1
            if (i + 1) % batch_size == 0:
                print(f"  Procesadas {i+1}/{total} imágenes...")

        except Exception as e:
            fallidas += 1
            print(f"  [WARN] Error en {img_data['url'][:60]}: {e}")

    print(f"\n[OK] Ingesta completada: {exitosas} exitosas, {fallidas} fallidas")
    print(f"Total en imagenes_embeddings: {images_col.count_documents({})}")


# Ejecutar la ingesta
ingest_images_from_recetas(IMAGES_TO_PROCESS)

/tmp/ipykernel_21706/3579506112.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha_ingesta": datetime.utcnow()


  Procesadas 10/50 imágenes...
  Procesadas 20/50 imágenes...
  Procesadas 30/50 imágenes...
  Procesadas 40/50 imágenes...
  Procesadas 50/50 imágenes...

[OK] Ingesta completada: 50 exitosas, 0 fallidas
Total en imagenes_embeddings: 50


## 9. Funciones de Búsqueda Vectorial Multimodal Cruzada

In [9]:
def _vector_search_pipeline(index_name, path, query_vector,
                             num_candidates=100, limit=5, filter_query=None):
    stage = {
        "$vectorSearch": {
            "index": index_name,
            "path": path,
            "queryVector": query_vector,
            "numCandidates": num_candidates,
            "limit": limit,
        }
    }
    if filter_query:
        stage["$vectorSearch"]["filter"] = filter_query
    return [stage]


def search_text_to_text(
    question: str,
    top_k: int = 5,
    dificultad: Optional[str] = None,
    max_calorias: Optional[int] = None,
    max_tiempo: Optional[int] = None,
    idioma: Optional[str] = None
) -> pd.DataFrame:
    qvec = embed_text_minilm(question)
    filters = []
    if dificultad:
        filters.append({"meta.dificultad": {"$eq": dificultad}})
    if max_calorias:
        filters.append({"meta.calorias": {"$lte": max_calorias}})
    if max_tiempo:
        filters.append({"meta.tiempo": {"$lte": max_tiempo}})
    if idioma:
        filters.append({"meta.idioma": {"$eq": idioma}})

    filter_query = None
    if filters:
        filter_query = {"$and": filters} if len(filters) > 1 else filters[0]

    pipeline = _vector_search_pipeline(
        VECTOR_INDEX_NAME_TEXT, "embedding", qvec,
        num_candidates=150, limit=top_k,
        filter_query=filter_query
    ) + [{
        "$project": {
            "_id": 0, "id_documento_fuente": 1, "chunk_id": 1,
            "titulo_receta": 1, "tipo_fuente": 1, "estrategia_chunking": 1,
            "texto_chunk": 1, "meta": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }]
    return pd.DataFrame(list(chunks_col.aggregate(pipeline)))


def search_text_to_image(question: str, top_k: int = 5) -> pd.DataFrame:
    qvec = embed_text_clip(question)
    pipeline = _vector_search_pipeline(
        VECTOR_INDEX_NAME_IMAGES, "embedding_clip", qvec, limit=top_k
    ) + [{
        "$project": {
            "_id": 0, "receta_id": 1, "titulo_receta": 1,
            "url": 1, "descripcion": 1, "etiquetas": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }]
    return pd.DataFrame(list(images_col.aggregate(pipeline)))


def search_image_to_image(image_source: str, top_k: int = 5, exclude_same: bool = True) -> pd.DataFrame:
    qvec = embed_image_clip(image_source)
    pipeline = _vector_search_pipeline(
        VECTOR_INDEX_NAME_IMAGES, "embedding_clip", qvec,
        limit=top_k + (1 if exclude_same else 0)
    ) + [{
        "$project": {
            "_id": 0, "receta_id": 1, "titulo_receta": 1,
            "url": 1, "descripcion": 1, "etiquetas": 1,
            "score": {"$meta": "vectorSearchScore"}
        }
    }]
    df = pd.DataFrame(list(images_col.aggregate(pipeline)))
    if exclude_same and not df.empty:
        df = df[df["url"] != image_source].head(top_k)
    return df.reset_index(drop=True)


def search_image_to_text(image_source: str, top_k: int = 5) -> pd.DataFrame:
    print("[INFO] search_image_to_text no disponible en esta version.")
    return pd.DataFrame()

## 10. Construcción de contexto multimodal y cliente LLM

In [10]:
def build_text_context(df: pd.DataFrame, max_chars: int = 4000) -> str:
    if df is None or df.empty:
        return ""
    blocks = []
    total = 0
    for i, row in df.iterrows():
        block = (
            f"[Fragmento Gastronomico {i+1}]\n"
            f"Receta: {row.get('titulo_receta', 'N/D')}\n"
            f"Estrategia: {row.get('estrategia_chunking', 'N/D')}\n"
            f"Score: {float(row.get('score', 0)):.4f}\n"
            f"Contenido: {row.get('texto_chunk', '')}"
        )
        if total + len(block) > max_chars:
            break
        blocks.append(block)
        total += len(block) + 2
    return "\n\n".join(blocks)


def build_image_context(df: pd.DataFrame, max_chars: int = 4000) -> str:
    if df is None or df.empty:
        return ""
    blocks = []
    total = 0
    for i, row in df.iterrows():
        etiquetas = ", ".join(row.get("etiquetas", [])) if isinstance(row.get("etiquetas", []), list) else ""
        block = (
            f"[Imagen Catalogo {i+1}]\n"
            f"Receta: {row.get('titulo_receta', 'N/D')}\n"
            f"Descripcion: {row.get('descripcion', '')}\n"
            f"Etiquetas: {etiquetas}\n"
            f"URL: {row.get('url', '')}\n"
            f"Score: {float(row.get('score', 0)):.4f}"
        )
        if total + len(block) > max_chars:
            break
        blocks.append(block)
        total += len(block) + 2
    return "\n\n".join(blocks)


def get_llm_client(api_key: str, base_url: str = LLM_BASE_URL):
    if not api_key:
        raise ValueError("No se proporciono API key para el LLM.")
    return OpenAI(api_key=api_key, base_url=base_url)


def answer_with_llm(client: OpenAI, task_description: str, context: str,
                    model: str = LLM_MODEL, temperature: float = 0.2, max_tokens: int = 700) -> str:
    system_prompt = (
        "Eres CookFlow-AI, un asistente de alta cocina experto en analisis multimodal. "
        "Responde unicamente con base en el contexto recuperado de MongoDB Atlas. "
        "Si el contexto no es suficiente, dilo explicitamente."
    )
    user_prompt = f"""
Consulta del usuario:
{task_description}

Contexto recuperado:
{context}

Instrucciones:
- Responde en español de forma clara.
- No inventes ingredientes ni URLs que no aparezcan en el contexto.
- Si faltan evidencias, indicalo explicitamente.
"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

## 11. Persistencia de consultas y resultados

In [11]:
def save_multimodal_run(mode: str, input_payload: dict, retrieved_df: pd.DataFrame, answer: Optional[str] = None) -> str:
    query_id = str(uuid.uuid4())

    # Consolidamos todo el documento de auditoría de forma estructurada e idempotente
    documento_auditoria = {
        "id_consulta": query_id,
        "tipo_registro": "auditoria_multimodal_cookflow",
        "fecha": datetime.utcnow(),
        "estrategia_flujo": mode,
        "input_payload": input_payload,
        "interaccion": {
            "question": input_payload.get("text_query") or input_payload.get("image_source"),
            "answer": answer if answer else "[Sin respuesta LLM]"
        },
        "soporte_recuperado": retrieved_df.to_dict(orient="records") if (retrieved_df is not None and not retrieved_df.empty) else []
    }

    # Inserción atómica y centralizada en tu colección real autorizada
    history_collection.insert_one(documento_auditoria)
    return query_id

## 12. Pipeline multimodal unificado

In [12]:
def multimodal_pipeline(
    mode: str,
    text_query: Optional[str] = None,
    image_source: Optional[str] = None,
    top_k: int = 5,
    dificultad: Optional[str] = None,
    max_calorias: Optional[int] = None,
    max_tiempo: Optional[int] = None,
    idioma: Optional[str] = None,
    max_context_chars: int = 3500,
    use_llm: bool = True,
) -> Dict[str, Any]:
    mode = mode.lower().strip()

    if mode == "text_to_text":
        if not text_query:
            raise ValueError("text_query es obligatorio para text_to_text")
        retrieved = search_text_to_text(
            text_query, top_k=top_k,
            dificultad=dificultad,
            max_calorias=max_calorias,
            max_tiempo=max_tiempo,
            idioma=idioma
        )
        context = build_text_context(retrieved, max_chars=max_context_chars)
        task_description = text_query

    elif mode == "text_to_image":
        if not text_query:
            raise ValueError("text_query es obligatorio para text_to_image")
        retrieved = search_text_to_image(text_query, top_k=top_k)
        context = build_image_context(retrieved, max_chars=max_context_chars)
        task_description = f"A partir de la consulta '{text_query}', describe las imagenes recuperadas y su relevancia."

    elif mode == "image_to_image":
        if not image_source:
            raise ValueError("image_source es obligatorio para image_to_image")
        retrieved = search_image_to_image(image_source, top_k=top_k)
        context = build_image_context(retrieved, max_chars=max_context_chars)
        task_description = (
            "El usuario busca platos visualmente similares a una imagen. "
            "El sistema CLIP ya encontro las imagenes mas similares. "
            "Basandote en las descripciones y etiquetas, explica que tienen "
            "en comun estos platos y por que son relevantes."
        )

    elif mode == "image_to_text":
        print("[INFO] image_to_text no disponible en esta version.")
        return {"query_id": None, "mode": mode, "retrieved": pd.DataFrame(),
                "context": "", "answer": None}

    else:
        raise ValueError("Modo no soportado. Usa: text_to_text, text_to_image, image_to_image")

    answer = None
    if use_llm:
        if not context.strip():
            answer = "No se recupero contexto suficiente para responder."
        else:
            client_llm = get_llm_client(GROQ_API_KEY, LLM_BASE_URL)
            answer = answer_with_llm(client_llm, task_description, context)

    query_id = save_multimodal_run(
        mode=mode,
        input_payload={
            "text_query": text_query,
            "image_source": image_source,
            "top_k": top_k,
            "filtros": {
                "dificultad": dificultad,
                "max_calorias": max_calorias,
                "max_tiempo": max_tiempo,
                "idioma": idioma
            }
        },
        retrieved_df=retrieved,
        answer=answer,
    )

    return {"query_id": query_id, "mode": mode, "retrieved": retrieved,
            "context": context, "answer": answer}

## 13. Consultas por modalidad

In [13]:
from IPython.display import HTML, display

print("="*60)
print("       COOKFLOW-AI — BÚSQUEDA MULTIMODAL")
print("="*60)
print("Selecciona el tipo de búsqueda:")
print("  1. Texto → Texto    (consulta con filtros opcionales)")
print("  2. Texto → Imagen   (encuentra imágenes por descripción)")
print("  3. Imagen → Imagen  (encuentra platos visualmente similares)")
print("="*60)

opcion = input("Ingresa el número (1/2/3): ").strip()

modos = {"1": "text_to_text", "2": "text_to_image", "3": "image_to_image"}
ejemplos = {
    "1": "recetas faciles con pollo",
    "2": "sopa colombiana tradicional",
    "3": "https://images.unsplash.com/photo-1569050467447-ce54b3bbc37d?w=800"
}

if opcion not in modos:
    print("Opcion no valida. Elige 1, 2 o 3.")
else:
    modo_seleccionado = modos[opcion]
    es_texto = opcion in ["1", "2"]

    print(f"\nModo: {modo_seleccionado}")
    print(f"Ejemplo: {ejemplos[opcion]}")
    entrada = input("Ingresa tu consulta (Enter para usar el ejemplo): ").strip()
    if not entrada:
        entrada = ejemplos[opcion]

    # Filtros hibridos solo para text_to_text
    dificultad_filtro = None
    max_cal_filtro = None
    max_tiempo_filtro = None

    if opcion == "1":
        print("\nFiltros opcionales (Enter para omitir):")
        d = input("  Dificultad [facil / media / dificil / experto]: ").strip()
        c = input("  Maximo de calorias (ej: 400): ").strip()
        t = input("  Maximo de tiempo en minutos (ej: 30): ").strip()
        dificultad_filtro = d if d else None
        max_cal_filtro    = int(c) if c.isdigit() else None
        max_tiempo_filtro = int(t) if t.isdigit() else None

        if dificultad_filtro or max_cal_filtro or max_tiempo_filtro:
            print(f"\n[Filtros activos] dificultad={dificultad_filtro}, "
                  f"max_calorias={max_cal_filtro}, max_tiempo={max_tiempo_filtro}")
        else:
            print("\n[Sin filtros — busqueda semantica pura]")

    result_multimodal = multimodal_pipeline(
        mode=modo_seleccionado,
        text_query=entrada if es_texto else None,
        image_source=None if es_texto else entrada,
        top_k=5,
        dificultad=dificultad_filtro,
        max_calorias=max_cal_filtro,
        max_tiempo=max_tiempo_filtro,
        use_llm=bool(GROQ_API_KEY)
    )

    print(f"\nID Auditoria: {result_multimodal['query_id']}")
    print("\nRESULTADOS RECUPERADOS:")

    df_visual = result_multimodal["retrieved"].copy()

    if df_visual is None or df_visual.empty:
        print("[INFO] No se recuperaron resultados.")
    elif "url" in df_visual.columns:
        df_visual["Vista Previa"] = df_visual["url"].apply(
            lambda url: f'<img src="{url}" width="120" style="border-radius:8px;"/>'
        )
        cols = ["Vista Previa"] + [c for c in df_visual.columns
                if c not in ["Vista Previa", "embedding_clip"]]
        display(HTML(df_visual[cols].to_html(escape=False)))
    else:
        cols = [c for c in df_visual.columns if c != "embedding_clip"]
        display(df_visual[cols])

    print("\nRESPUESTA DE COOKFLOW-AI:")
    print("="*70)
    print(result_multimodal["answer"] if result_multimodal["answer"] else "[Sin LLM]")
    print("="*70)

       COOKFLOW-AI — BÚSQUEDA MULTIMODAL
Selecciona el tipo de búsqueda:
  1. Texto → Texto    (consulta con filtros opcionales)
  2. Texto → Imagen   (encuentra imágenes por descripción)
  3. Imagen → Imagen  (encuentra platos visualmente similares)


KeyboardInterrupt: Interrupted by user

## 14. Funciones prácticas de acceso rápido

In [14]:
def ask_text(question: str, top_k: int = 5, use_llm: bool = True):
    return multimodal_pipeline("text_to_text", text_query=question, top_k=top_k, use_llm=use_llm)


def ask_images(question: str, top_k: int = 5, use_llm: bool = True):
    return multimodal_pipeline("text_to_image", text_query=question, top_k=top_k, use_llm=use_llm)


def find_similar_images(image_source: str, top_k: int = 5, use_llm: bool = True):
    return multimodal_pipeline("image_to_image", image_source=image_source, top_k=top_k, use_llm=use_llm)


def explain_image_with_text(image_source: str, top_k: int = 5, use_llm: bool = True):
    return multimodal_pipeline("image_to_text", image_source=image_source, top_k=top_k, use_llm=use_llm)

## 15. Explorar lo que quedó guardado

In [15]:
print("=== ÚLTIMOS LOGS DE AUDITORÍA MULTIMODAL REGISTRADOS ===")

logs_multimodales = list(history_collection.find(
    {"tipo_registro": "auditoria_multimodal_cookflow"},
    {"_id": 0, "id_consulta": 1, "estrategia_flujo": 1, "interaccion.question": 1, "interaccion.answer": 1, "fecha": 1}
).sort("fecha", -1).limit(5))

if logs_multimodales:
    df_logs_mm = pd.DataFrame([{
        "ID Consulta": l.get("id_consulta"),
        "Fecha UTC": l.get("fecha"),
        "Estrategia Multimodal": l.get("estrategia_flujo"),
        "Input Evaluado": l.get("interaccion", {}).get("question")[:50] + "..." if l.get("interaccion", {}).get("question") else "N/D",
        "Respuesta LLM": l.get("interaccion", {}).get("answer")[:100] + "..." if l.get("interaccion", {}).get("answer") else "N/D"
    } for l in logs_multimodales])
    display(df_logs_mm)
else:
    print("[INFO] No se registran logs multimediales recientes en 'auditoria_rag'. Ejecuta la prueba interactiva de arriba.")

=== ÚLTIMOS LOGS DE AUDITORÍA MULTIMODAL REGISTRADOS ===


,ID Consulta,Fecha UTC,Estrategia Multimodal,Input Evaluado,Respuesta LLM
0,1e5460cc-b5b0-4e37-b898-210b16265af1,2026-06-11 00:29:24.820,text_to_text,Recetas saludables valoradas con mas de 4 estr...,"Basándome en el contexto recuperado, puedo ide..."
1,ec6d80a4-69a6-4e8b-a7a7-b9da4fe53177,2026-06-11 00:29:15.756,text_to_text,"Que puedo cocinar con zanahoria, cebolla y ajo...","Con base en el contexto recuperado, puedo suge..."
2,18dad20f-cf8e-46cd-97ec-8f936ae39ba7,2026-06-11 00:29:04.496,text_to_text,Algo facil para sorprender a mis invitados...,"¡Claro! Al parecer, el usuario está buscando u..."
3,ee1be295-83d6-4d7b-86de-462e2d42eb67,2026-06-11 00:28:56.956,text_to_text,Recetas tipicas de Colombia para una cena espe...,"¡Claro! Basándome en el contexto recuperado, p..."
4,d59f7454-4fb6-44f7-97dd-77b615517336,2026-06-11 00:28:46.794,text_to_text,Platos vegetarianos ricos en proteinas...,"Basándome en el contexto recuperado, puedo sug..."


## 16. Instalación de RAGAS

In [16]:
!pip install requests==2.32.4
print("[OK] RAGAS instalado")

[OK] RAGAS instalado


## 17. Dataset de evaluación RAGAS

In [17]:
EVAL_DATASET = [
    {
        "question": "Recetas con pollo y menos de 400 calorias",
        "ground_truth": "Recetas como el Pollo al Curry Express con 350 kcal o el Ceviche de Camaron con 180 kcal son opciones con pollo o proteina ligera por debajo de 400 calorias."
    },
    {
        "question": "Como se hace una buena salsa de tomate casera?",
        "ground_truth": "Una buena salsa de tomate requiere sofreir cebolla y ajo, agregar tomates frescos maduros, cocinar a fuego lento y rectificar con sal y azucar."
    },
    {
        "question": "Opciones sin gluten para el desayuno",
        "ground_truth": "El Ajiaco Bogotano y el Ceviche de Camaron tienen etiqueta sin-gluten. Para desayuno, opciones naturalmente sin gluten incluyen preparaciones con arroz, frutas o huevos."
    },
    {
        "question": "Recetas faciles para principiantes con arroz",
        "ground_truth": "El Arroz con Leche Colombiano y el Arroz Chaufa Peruano son recetas de dificultad facil que usan arroz como ingrediente principal."
    },
    {
        "question": "Que postre puedo hacer en menos de 20 minutos?",
        "ground_truth": "El Arroz con Leche Colombiano se puede preparar en 25 minutos. Para menos de 20 minutos, opciones rapidas incluyen frutas con crema o helado casero simple."
    },
    {
        "question": "Platos vegetarianos ricos en proteinas",
        "ground_truth": "El Hummus Casero Autentico y la Sopa de Miso con Tofu son opciones vegetarianas con buena fuente de proteinas vegetales."
    },
    {
        "question": "Recetas tipicas de Colombia para una cena especial",
        "ground_truth": "El Ajiaco Bogotano, la Bandeja Paisa Tradicional y el Sancocho de Gallina Costeno son platos emblematicos colombianos ideales para cenas especiales."
    },
    {
        "question": "Algo facil para sorprender a mis invitados",
        "ground_truth": "La Sopa de Miso con Tofu es facil de preparar en 20 minutos y sorprende por su sabor exotico. El Gazpacho Andaluz tambien es sencillo y elegante."
    },
    {
        "question": "Que puedo cocinar con zanahoria, cebolla y ajo?",
        "ground_truth": "Con zanahoria, cebolla y ajo se puede preparar el Borsch Ucraniano, sopas de verduras o un sofrito base para multiples platos."
    },
    {
        "question": "Recetas saludables valoradas con mas de 4 estrellas",
        "ground_truth": "El Ceviche de Camaron con 180 kcal y el Gazpacho Andaluz con 95 kcal son opciones saludables y bien valoradas en la plataforma."
    },
]

print(f"Dataset de evaluacion preparado: {len(EVAL_DATASET)} pares pregunta-ground_truth")

Dataset de evaluacion preparado: 10 pares pregunta-ground_truth


## 18. Ejecutar pipeline sobre dataset de evaluacion

In [18]:
from datasets import Dataset

print("Ejecutando pipeline RAG sobre las 10 consultas de evaluacion...")
print("Esto puede tardar 1-2 minutos...\n")

questions    = []
answers      = []
contexts     = []
ground_truths = []

for i, item in enumerate(EVAL_DATASET):
    print(f"  Consulta {i+1}/10: {item['question'][:50]}...")
    try:
        result = multimodal_pipeline(
            mode="text_to_text",
            text_query=item["question"],
            top_k=5,
            use_llm=True
        )
        questions.append(item["question"])
        answers.append(result["answer"] if result["answer"] else "")
        contexts.append([result["context"]] if result["context"] else [""])
        ground_truths.append(item["ground_truth"])
    except Exception as e:
        print(f"  [WARN] Error en consulta {i+1}: {e}")
        questions.append(item["question"])
        answers.append("")
        contexts.append([""])
        ground_truths.append(item["ground_truth"])

eval_dataset = Dataset.from_dict({
    "question":    questions,
    "answer":      answers,
    "contexts":    contexts,
    "ground_truth": ground_truths,
})

print(f"\n[OK] Dataset de evaluacion construido: {len(eval_dataset)} filas")

Ejecutando pipeline RAG sobre las 10 consultas de evaluacion...
Esto puede tardar 1-2 minutos...

  Consulta 1/10: Recetas con pollo y menos de 400 calorias...


/tmp/ipykernel_21706/1593164907.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha": datetime.utcnow(),


  Consulta 2/10: Como se hace una buena salsa de tomate casera?...


/tmp/ipykernel_21706/1593164907.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha": datetime.utcnow(),


  Consulta 3/10: Opciones sin gluten para el desayuno...


...

/tmp/ipykernel_21706/1593164907.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha": datetime.utcnow(),


  Consulta 9/10: Que puedo cocinar con zanahoria, cebolla y ajo?...


/tmp/ipykernel_21706/1593164907.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha": datetime.utcnow(),


  Consulta 10/10: Recetas saludables valoradas con mas de 4 estrella...

[OK] Dataset de evaluacion construido: 10 filas


/tmp/ipykernel_21706/1593164907.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fecha": datetime.utcnow(),


## 19. Evaluacion con RAGAS

In [20]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from google.colab import userdata
import os

# 1. Forzar las variables de entorno para OpenRouter
os.environ["OPENAI_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

api_key = os.environ["OPENAI_API_KEY"]
api_base = os.environ["OPENAI_API_BASE"]

# 2. Configurar el LLM con DeepSeek V3
base_llm = ChatOpenAI(
    model="deepseek/deepseek-chat",
    openai_api_key=api_key,
    openai_api_base=api_base,
    temperature=0,
    max_tokens=1000
)
ragas_llm = LangchainLLMWrapper(base_llm)

# 3. Configurar Embeddings a través de OpenRouter para apagar la llamada automática a OpenAI
base_embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=api_key,
    openai_api_base=api_base
)
ragas_embeddings = LangchainEmbeddingsWrapper(base_embeddings)

# 4. Asignar de forma estricta los componentes a las métricas
faithfulness.llm = ragas_llm

answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings # <--- Esto soluciona los 10 errores 401 de raíz

print("Ejecutando evaluacion RAGAS corregida y completa en OpenRouter...")
print("Esto puede tardar 1-2 minutos...\n")

try:
    # Pasamos los objetos globales también al evaluador para asegurar
    result_ragas = evaluate(
        dataset=eval_dataset,
        metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=ragas_embeddings
    )

    df_ragas = result_ragas.to_pandas()
    print("\n=== RESULTADOS RAGAS ===")
    display(df_ragas[["question", "faithfulness", "answer_relevancy"]])
    print(f"\nFaithfulness promedio:      {df_ragas['faithfulness'].mean():.3f}")
    print(f"Answer Relevancy promedio:  {df_ragas['answer_relevancy'].mean():.3f}")

except Exception as e:
    print(f"[ERROR] RAGAS fallo: {e}")

Ejecutando evaluacion RAGAS corregida y completa en OpenRouter...
Esto puede tardar 1-2 minutos...



Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


=== RESULTADOS RAGAS ===


,question,faithfulness,answer_relevancy
0,Recetas con pollo y menos de 400 calorias,0.727273,0.000000
1,Como se hace una buena salsa de tomate casera?,0.444444,0.000000
2,Opciones sin gluten para el desayuno,0.181818,0.000000
3,Recetas faciles para principiantes con arroz,0.176471,0.745280
4,Que postre puedo hacer en menos de 20 minutos?,0.571429,0.000000
5,Platos vegetarianos ricos en proteinas,0.583333,0.544721
6,Recetas tipicas de Colombia para una cena espe...,0.777778,0.000000
7,Algo facil para sorprender a mis invitados,0.909091,0.594207
8,"Que puedo cocinar con zanahoria, cebolla y ajo?",0.375000,0.000000
9,Recetas saludables valoradas con mas de 4 estr...,0.789474,0.000000



Faithfulness promedio:      0.554
Answer Relevancy promedio:  0.188


## 20. Guardar resultados RAGAS en MongoDB

In [22]:
from datetime import datetime, timezone

evaluaciones_col = db["evaluaciones_rag"]

registros = []
for _, row in df_ragas.iterrows():
    registros.append({
        "question":         row["question"],
        "faithfulness":     float(row["faithfulness"]) if pd.notna(row["faithfulness"]) else None,
        "answer_relevancy": float(row["answer_relevancy"]) if pd.notna(row["answer_relevancy"]) else None,
        "modelo_eval":      "ragas-faithfulness-answer_relevancy",
        "llm_evaluador":    "deepseek/deepseek-chat via OpenRouter",
        "embeddings_eval":  "openai/text-embedding-3-small via OpenRouter",
        "llm_pipeline":     "llama-3.1-8b-instant via Groq",
        "fecha":            datetime.now(timezone.utc)
    })

evaluaciones_col.insert_many(registros)
print(f"[OK] {len(registros)} evaluaciones guardadas en evaluaciones_rag")
print(f"Total en evaluaciones_rag: {evaluaciones_col.count_documents({})}")

print("\n=== RESUMEN FINAL GUARDADO EN MONGODB ===")
df_guardado = pd.DataFrame(list(evaluaciones_col.find(
    {}, {"_id": 0, "question": 1, "faithfulness": 1, "answer_relevancy": 1}
).sort("faithfulness", -1)))
display(df_guardado)

print(f"\nFaithfulness promedio:     {df_guardado['faithfulness'].mean():.3f}")
print(f"Answer Relevancy promedio: {df_guardado['answer_relevancy'].mean():.3f}")

[OK] 10 evaluaciones guardadas en evaluaciones_rag
Total en evaluaciones_rag: 10

=== RESUMEN FINAL GUARDADO EN MONGODB ===


,question,faithfulness,answer_relevancy
0,Algo facil para sorprender a mis invitados,0.909091,0.594207
1,Recetas saludables valoradas con mas de 4 estr...,0.789474,0.000000
2,Recetas tipicas de Colombia para una cena espe...,0.777778,0.000000
3,Recetas con pollo y menos de 400 calorias,0.727273,0.000000
4,Platos vegetarianos ricos en proteinas,0.583333,0.544721
5,Que postre puedo hacer en menos de 20 minutos?,0.571429,0.000000
6,Como se hace una buena salsa de tomate casera?,0.444444,0.000000
7,"Que puedo cocinar con zanahoria, cebolla y ajo?",0.375000,0.000000
8,Opciones sin gluten para el desayuno,0.181818,0.000000
9,Recetas faciles para principiantes con arroz,0.176471,0.745280



Faithfulness promedio:     0.554
Answer Relevancy promedio: 0.188
